# Lab Day 2 — Backbone, công thức huấn luyện và suy luận trên DeepWeeds

Notebook **khung** cho Colab hoặc Kaggle. Các ô có chữ `TODO` là phần **bạn tự viết**; ô cài đặt, tải dữ liệu
và chạy `eval.py` đã viết sẵn. Đọc `README.md` (quy tắc chia dữ liệu, cách đánh giá), `GUIDE.md` và `RUBRIC.md` trước.

**Quy tắc quan trọng nhất:** chọn mọi thứ trên **val**; **test chỉ chạy một lần mỗi seed ở Bước 4**.

Bản đồ: Bước 0 (EDA, kiểm tra) → Bước 1 (backbone) → Bước 2 (công thức huấn luyện) → Bước 3 (suy luận)
→ Bước 4 (chung kết + `eval.py`) → Bước 5 (xlsx, biểu đồ, báo cáo).

## 0. Cài đặt môi trường

In [ ]:
# Cài thư viện (Colab/Kaggle). Kaggle: bật Internet trong Settings.
!pip -q install timm openpyxl

import sys, os, platform
import numpy as np, pandas as pd, torch, timm

# Đã `git clone` repo bài lab vào thư mục này (đổi nếu bạn đặt chỗ khác).
REPO_DIR = "K4-Track4-Day2-Deeplearning-Advance"
sys.path.insert(0, REPO_DIR)                 # để import eval.py
sys.path.insert(0, f"{REPO_DIR}/starter")    # hoặc thư mục code/ của bạn sau khi chép starter vào

print("python", platform.python_version(), "| torch", torch.__version__, "| timm", timm.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU")

### Tải dữ liệu (đã viết sẵn)
Ảnh từ Zenodo (~490 MB, kiểm tra MD5), nhãn và các file chia fold từ GitHub của tác giả.
**Sau khi giải nén, hãy `ls` để xem ảnh nằm ở thư mục nào** rồi đặt `IMAGES_DIR` cho đúng.

In [ ]:
import hashlib

os.makedirs("data/labels", exist_ok=True)
if not os.path.exists("data/images.zip"):
    !wget -q -O data/images.zip "https://zenodo.org/records/7939060/files/images.zip?download=1"

h = hashlib.md5()
with open("data/images.zip", "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        h.update(chunk)
assert h.hexdigest() == "b7b30f96d466fba86016aa5a26606e0f", f"MD5 sai: {h.hexdigest()}"

!unzip -q -n data/images.zip -d data/
!ls data | head

BASE = "https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels"
for name in ["labels", "train_subset0", "val_subset0", "test_subset0"]:
    !wget -q -O data/labels/{name}.csv {BASE}/{name}.csv
!ls -la data/labels

In [ ]:
IMAGES_DIR = "data/images"   # TODO: kiểm tra đúng thư mục chứa các file .jpg sau khi giải nén
LABELS_DIR = "data/labels"

## Bước 0 — EDA và kiểm tra pipeline
Mục tiêu: hiểu dữ liệu, chạy các kiểm tra chia dữ liệu (README mục 2.1), kiểm tra pipeline trước khi chạy thật.

In [ ]:
# 1. Đọc và kiểm tra split (S1-S6)
import dataset
train_df, val_df, test_df = dataset.load_split(LABELS_DIR, fold=0)
split_info = dataset.check_split(train_df, val_df, test_df, IMAGES_DIR)

# In bảng số lượng chi tiết từng lớp
import pandas as pd
df_counts = pd.DataFrame(split_info["per_class"]).T[["train", "val", "test", "total"]]
print(df_counts)

# 2. EDA: Phân tích phân bố lớp
import matplotlib.pyplot as plt, numpy as np
from PIL import Image
plt.figure(figsize=(12, 5), dpi=120)
x = np.arange(len(dataset.CLASS_NAMES))
w = 0.25
plt.bar(x - w, df_counts["train"], width=w, label="Train (~60%)", color="#3498db")
plt.bar(x, df_counts["val"], width=w, label="Val (~20%)", color="#2ecc71")
plt.bar(x + w, df_counts["test"], width=w, label="Test (~20%)", color="#e67e22")
plt.xticks(x, dataset.CLASS_NAMES, rotation=35, ha="right")
plt.ylabel("Số lượng ảnh")
plt.title("Phân bố lớp DeepWeeds (Fold 0)")
plt.legend()
plt.grid(axis="y", linestyle=":", alpha=0.6)
plt.tight_layout()
plt.show()

# Tỉ lệ mất cân bằng
max_cnt = df_counts["total"].max()
min_cnt = df_counts["total"].min()
max_name = df_counts[df_counts["total"] == max_cnt].index[0]
min_name = df_counts[df_counts["total"] == min_cnt].index[0]
print(f"Lớp nhiều nhất: '{max_name}' ({max_cnt} ảnh)")
print(f"Lớp ít nhất:    '{min_name}' ({min_cnt} ảnh)")
print(f"Tỉ lệ mất cân bằng: {max_cnt / min_cnt:.2f}x (Đối chiếu Table 1: Negatives 9.106 ảnh, các loài 1.009-1.125 ảnh)")


In [ ]:
# Kiểm tra pipeline (slide trang 59, GUIDE.md mục 1.3):
import math, torch, torch.nn.functional as F
import model as mod, losses, train

# 1) Cố định seed
train.set_seed(42)
print("1) Seed cố định thành công.")

# 2) Loss ban đầu xấp xỉ -ln(1/9) ≈ 2.1972
m = mod.build_model("resnet50", pretrained=False, num_classes=9, init="finetune")
m.eval()
with torch.no_grad():
    init_loss = F.cross_entropy(m(torch.randn(64, 3, 224, 224)), torch.randint(0, 9, (64,))).item()
print(f"2) Loss ban đầu: {init_loss:.4f} (Kỳ vọng: {-math.log(1/9):.4f})")

# 3) Overfit một batch nhỏ
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
m_overfit = mod.build_model("resnet50", pretrained=False, num_classes=9).to(device)
m_overfit.train()
opt = torch.optim.Adam(m_overfit.parameters(), lr=1e-3)
bx = torch.randn(8, 3, 224, 224, device=device)
by = torch.tensor([0, 1, 2, 3, 4, 5, 6, 7], device=device)
for _ in range(60):
    opt.zero_grad()
    loss = F.cross_entropy(m_overfit(bx), by)
    loss.backward()
    opt.step()
print(f"3) Loss overfit sau 60 step: {loss.item():.6f} (< 0.05 -> THÀNH CÔNG)")

# 4) Vẽ ảnh sau augmentation cùng nhãn
tf = dataset.build_transforms(train=True, img_size=224, aug="basic")
ds = dataset.DeepWeedsDataset(train_df.iloc[:4], IMAGES_DIR, transform=tf)
fig, axes = plt.subplots(1, 4, figsize=(12, 3), dpi=100)
for i in range(4):
    img_t, lbl, fn = ds[i]
    img_np = img_t.numpy().transpose(1, 2, 0) * np.array(dataset.IMAGENET_STD) + np.array(dataset.IMAGENET_MEAN)
    axes[i].imshow(np.clip(img_np, 0, 1))
    axes[i].set_title(dataset.CLASS_NAMES[lbl])
    axes[i].axis("off")
plt.tight_layout()
plt.show()
print("4) Ảnh sau augmentation khớp nhãn.")


## Bước 1 — So sánh backbone (≥ 5)
Cùng công thức nền `T00` (GUIDE.md mục 1.4), cùng seed, mỗi backbone một lần chạy. Mã thí nghiệm `B01`, `B02`, ...

In [ ]:
# TODO: vòng lặp qua danh sách backbone, mỗi lần gọi train.run(Config(exp_id="B01", backbone=..., seed=0))
#       Gợi ý: from train import Config, run
# TODO: ghi vào bảng: tag trọng số, #params, GMAC, macro-F1 val, top-1 val, thời gian train/epoch.
# TODO: chọn 1-2 backbone đi tiếp, kèm lý do dựa trên số liệu.

## Bước 2 — Công thức huấn luyện (≥ 3 trục)
Mỗi lần chạy chỉ khác `T00` **một yếu tố**. Mã `T01`, `T02`, ... Ghi rõ khác ở điểm nào.

In [ ]:
# TODO: các trục A-G của GUIDE.md mục 3 (khởi tạo, augmentation, loss, sampler, LR/optimizer, EMA, ...).
# TODO: thử ít nhất một kết hợp các yếu tố tốt; so Δ với nhiễu (std).

## Bước 3 — Suy luận (≥ 4 phương pháp) và độ trễ
Làm trên **val**. Không huấn luyện lại. Mã `I00` (1 view, mốc), `I01`, ...

In [ ]:
# TODO: TTA lật, multi-crop/scale, dò độ phân giải, gộp xác suất vs logit, ensemble, EMA/soup,
#       temperature scaling (T khớp trên VAL), gộp BN/FP16  (starter/inference.py)
# TODO: độ trễ p50/p95/p99 bằng starter/benchmark.py: warmup, synchronize, >= 50 lần

## Bước 4 — Chung kết: ≥ 3 seed, test một lần mỗi seed
1. Chốt cấu hình trên **val**.
2. Chạy cấu hình cuối **và mốc** (`T00` + `I00`) với cùng số seed, bật `save_test_predictions=True`.
3. Dự đoán lưu ở `predictions/<exp_id>_seed<k>_test.csv`. Rồi chạy `eval.py` (ô dưới).

In [ ]:
# TODO: chạy chung kết và mốc, mỗi seed, ví dụ:
#   for seed in (0, 1, 2):
#       run(Config(exp_id="F01", ..., seed=seed, save_test_predictions=True))
#       run(Config(exp_id="T00", seed=seed, save_test_predictions=True))
# Với suy luận (TTA/ensemble/temperature scaling), tự ghi predictions/F01_seed<k>_test.csv
# bằng eval.save_predictions(...) từ xác suất cuối cùng; thêm F01uncal_* nếu muốn chấm I4(a).

In [ ]:
# Tính chỉ số theo đúng định nghĩa của lớp (đã viết sẵn, không sửa eval.py):
!python {REPO_DIR}/eval.py score --pred "predictions/F01_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag F01 --out eval_out
!python {REPO_DIR}/eval.py score --pred "predictions/T00_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag T00 --out eval_out

In [ ]:
# Tự chấm phần I của RUBRIC (đề xuất, giảng viên xác nhận). Thêm --uncal, --final-val, --latency-p95-ms nếu có.
!python {REPO_DIR}/eval.py grade --final "predictions/F01_seed*_test.csv" \
    --baseline "predictions/T00_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --out eval_out

## Bước 5 — Sản phẩm
`results.xlsx` (các sheet ở GUIDE.md mục 6.1), `curves/<exp_id>_*.png`, `report.md`, `predictions/`, `code/`.

In [ ]:
# TODO: ghi results.xlsx bằng pandas.ExcelWriter(engine="openpyxl") với các sheet:
#   Backbones, Training, Inference, Final, PerClass, Latency, Summary
# TODO: kiểm tra mỗi exp_id có ảnh trong curves/ và số trong xlsx khớp kết quả của eval.py